# 영어 → 숫자 / 김한나

구조 변경: 기존 Encoder와 Seq2Seq에 `bidirectional=False/True` 옵션과 양방향 출력·hidden projection을 추가.

## 실험 방법
- 원본 데이터와 CRC32 8:1:1 분할을 사용합니다. 숫자는 문자열입니다.
- 동일 seed, 데이터 순서, 학습 설정으로 단방향과 양방향 GRU를 각각 학습합니다.
- 각 실험의 best validation exact match 체크포인트를 복원합니다.
- 최종 모델은 **validation 정확도**로 선택하고, test는 최종 비교 보고에만 사용합니다.
- 양방향 모델은 파라미터 수가 더 크므로 파라미터 수와 실행 시간을 함께 보고합니다.
- 기본값: embedding 128, hidden 256, 2층, dropout 0.2, 15 epochs, batch 128.
- 제출: `work/en2num.ipynb`, `en2num.pt`. 통합 시 아래 Encoder/Seq2Seq 변경을 공통 클래스에 반영해야 합니다.

## 구현 설명
양방향 encoder는 문장을 앞→뒤와 뒤→앞으로 읽습니다. 출력의 두 방향을 연결하면 2H가 되므로 Linear(2H,H)로 줄여 기존 Luong attention에 전달합니다. 각 층의 마지막 hidden도 두 방향을 연결하고 Linear+tanh로 H로 줄여 decoder를 초기화합니다. LSTM에서는 cell state에도 별도 projection을 적용합니다. 단방향에서는 projection 모듈이 생기지 않아 기존 체크포인트와 호환됩니다.


In [ ]:
# 설정
SRC, TGT = 'en', 'num'  # num, en, ko

DATA_DIR = '../data'
SAVE_DIR = '..'  # pt 저장 위치

CONFIG = dict(
    emb_dim=128,
    hid_dim=256,
    n_layers=2,
    dropout=0.2,
    rnn='gru',  # gru / lstm
    bidirectional=False,
    attention='luong',  # 'luong' / None
)
BATCH_SIZE = 128
EPOCHS = 15
LR = 1e-3
TEACHER_FORCING = 0.5
BEAM_SIZE = 5
SEED = 42
MAX_TRAIN = None  # 디버깅용 train 샘플 수

In [ ]:
# Colab: 런타임 > 런타임 유형 변경 > GPU 선택 후 모두 실행.
# 로컬에서는 project01/work 폴더를 작업 디렉터리로 사용합니다.
import sys
if 'google.colab' in sys.modules:
    import os, subprocess
    from pathlib import Path
    repo = Path('/content/2026-nlp')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--branch', 'en2num',
                        'https://github.com/jungyy808/2026-nlp.git', str(repo)], check=True)
    os.chdir(repo / 'project01/work')


In [ ]:
import os, random, time, zlib, math
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def set_seed(seed):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

torch.set_num_threads(min(4, os.cpu_count() or 1))
set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

## 데이터

In [ ]:
FILES = {'en': 'num2txt.txt', 'ko': 'num2kor.txt'}
PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ['<pad>', '<sos>', '<eos>', '<unk>']


def tokenize(text, lang):
    # en: 단어 단위 / num, ko: 문자 단위 (공백 포함)
    return text.split() if lang == 'en' else list(text)


def detokenize(tokens, lang):
    return ' '.join(tokens) if lang == 'en' else ''.join(tokens)


def load_pairs(path):
    # CRLF 제거
    pairs = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\r\n')
            if not line:
                continue
            num, text = line.split('\t', 1)
            pairs.append((num.strip(), text.strip()))
    return pairs


def get_pairs(src_lang, tgt_lang, data_dir=DATA_DIR):
    # (src, tgt, key) 반환, key는 split 기준
    assert src_lang != tgt_lang and {src_lang, tgt_lang} <= {'num', 'en', 'ko'}
    tables = {lang: dict(load_pairs(os.path.join(data_dir, FILES[lang])))
              for lang in (src_lang, tgt_lang) if lang != 'num'}
    if 'num' in (src_lang, tgt_lang):
        other = tgt_lang if src_lang == 'num' else src_lang
        keys = list(tables[other])
    else:  # en-ko: 공통 key만 사용
        keys = sorted(set(tables['en']) & set(tables['ko']))

    def text(lang, k):
        return k if lang == 'num' else tables[lang][k]

    return [(text(src_lang, k), text(tgt_lang, k), k) for k in keys]


def split_of(key):
    # key 해시 기준 8:1:1 분할
    b = zlib.crc32(key.encode('utf-8')) % 10
    return 'test' if b == 0 else 'valid' if b == 1 else 'train'


class Vocab:
    def __init__(self, lang, sentences=()):
        self.lang = lang
        tokens = sorted({t for s in sentences for t in tokenize(s, lang)})
        self.itos = SPECIALS + tokens
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    @classmethod
    def from_itos(cls, lang, itos):
        v = cls(lang)
        v.itos = list(itos)
        v.stoi = {t: i for i, t in enumerate(v.itos)}
        return v

    def encode(self, text):  # sos/eos 제외
        return [self.stoi.get(t, UNK) for t in tokenize(text, self.lang)]

    def decode(self, ids):
        out = []
        for i in ids:
            i = int(i)
            if i == EOS:
                break
            if i in (PAD, SOS):
                continue
            out.append(self.itos[i])
        return detokenize(out, self.lang)

    def __len__(self):
        return len(self.itos)


class PairDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.pairs = pairs
        self.src = [src_vocab.encode(s) for s, _, _ in pairs]
        self.tgt = [[SOS] + tgt_vocab.encode(t) + [EOS] for _, t, _ in pairs]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        return self.src[i], self.tgt[i]


def collate(batch):
    srcs, tgts = zip(*batch)
    src_len = torch.tensor([len(s) for s in srcs])
    src = torch.full((len(srcs), max(src_len)), PAD, dtype=torch.long)
    tgt = torch.full((len(tgts), max(len(t) for t in tgts)), PAD, dtype=torch.long)
    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src[i, :len(s)] = torch.tensor(s)
        tgt[i, :len(t)] = torch.tensor(t)
    return src, src_len, tgt


def make_loaders(pairs, src_lang, tgt_lang, batch_size=BATCH_SIZE, max_train=None, extra_train=None):
    # extra_train: 증강 등으로 추가할 (src, tgt, key) 리스트. train에만 들어가고 valid/test는 원본만 사용
    src_vocab = Vocab(src_lang, [s for s, _, _ in pairs])
    tgt_vocab = Vocab(tgt_lang, [t for _, t, _ in pairs])
    parts = defaultdict(list)
    for p in pairs:
        parts[split_of(p[2])].append(p)
    if max_train:
        parts['train'] = random.Random(SEED).sample(parts['train'], min(max_train, len(parts['train'])))
    if extra_train:
        held_out = {p[2] for p in parts['valid'] + parts['test']}
        extra = [p for p in extra_train if p[2] not in held_out]  # valid/test 숫자와 겹치는 건 제외
        print(f'extra_train: {len(extra):,}개 추가 (valid/test와 겹쳐서 제외 {len(extra_train) - len(extra):,}개)')
        parts['train'] = parts['train'] + extra
    loaders = [DataLoader(PairDataset(parts[name], src_vocab, tgt_vocab), batch_size=batch_size,
                          shuffle=(name == 'train'), collate_fn=collate)
               for name in ('train', 'valid', 'test')]
    return (*loaders, src_vocab, tgt_vocab)

In [ ]:
print('작업 위치:', os.getcwd())
for f in FILES.values():
    assert os.path.exists(os.path.join(DATA_DIR, f)), f'{os.path.abspath(os.path.join(DATA_DIR, f))} 없음 (Colab이면 clone 셀 확인)'
print('pt 저장 위치:', os.path.abspath(SAVE_DIR))

pairs = get_pairs(SRC, TGT)
train_loader, valid_loader, test_loader, src_vocab, tgt_vocab = make_loaders(
    pairs, SRC, TGT, BATCH_SIZE, MAX_TRAIN)

MAX_LEN = max(len(tokenize(t, TGT)) for _, t, _ in pairs) + 10  # 데이터 기준 최대 길이
print(f'{SRC} -> {TGT}: 전체 {len(pairs):,}쌍 | train {len(train_loader.dataset):,} / '
      f'valid {len(valid_loader.dataset):,} / test {len(test_loader.dataset):,}')
print(f'vocab: src {len(src_vocab)}, tgt {len(tgt_vocab)} | 디코딩 최대 길이 {MAX_LEN}')
for s, t, _ in pairs[:3]:
    print(f'  {s!r} -> {t!r}')

## 모델 (GRU/LSTM encoder-decoder + attention)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn, bidirectional=False):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        RNN = nn.LSTM if rnn == 'lstm' else nn.GRU
        self.rnn = RNN(emb_dim, hid_dim, n_layers, batch_first=True,
                       dropout=dropout if n_layers > 1 else 0, bidirectional=bidirectional)
        # 김한나: 두 방향의 2H 표현을 기존 decoder/attention의 H 차원으로 연결.
        self.bidirectional = bidirectional
        self.n_layers, self.hid_dim = n_layers, hid_dim
        if bidirectional:
            self.output_bridge = nn.Linear(2 * hid_dim, hid_dim)
            self.hidden_bridge = nn.Linear(2 * hid_dim, hid_dim)
            if rnn == 'lstm':
                self.cell_bridge = nn.Linear(2 * hid_dim, hid_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, src, src_len):
        x = self.drop(self.emb(src))
        packed = nn.utils.rnn.pack_padded_sequence(x, src_len.cpu(), batch_first=True, enforce_sorted=False)
        out, hidden = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=src.size(1))
        if self.bidirectional:
            out = self.output_bridge(out)
            def bridge(h, projection):
                # [layers*2,B,H] -> [layers,B,2H] -> [layers,B,H]
                h = h.reshape(self.n_layers, 2, h.size(1), self.hid_dim)
                return torch.tanh(projection(torch.cat((h[:, 0], h[:, 1]), dim=-1)))
            if isinstance(hidden, tuple):
                hidden = (bridge(hidden[0], self.hidden_bridge),
                          bridge(hidden[1], self.cell_bridge))
            else:
                hidden = bridge(hidden, self.hidden_bridge)
        return out, hidden


class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn, attention):
        super().__init__()
        assert attention in ('luong', None), f'지원하지 않는 attention: {attention}'  # 옵션 추가 시 여기에도 추가
        assert rnn in ('gru', 'lstm'), f'지원하지 않는 rnn: {rnn}'
        self.attention = attention
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        RNN = nn.LSTM if rnn == 'lstm' else nn.GRU
        self.rnn = RNN(emb_dim, hid_dim, n_layers, batch_first=True,
                       dropout=dropout if n_layers > 1 else 0)
        if attention == 'luong':
            self.attn = nn.Linear(hid_dim, hid_dim, bias=False)  # Luong (general)
            self.combine = nn.Linear(hid_dim * 2, hid_dim)
        self.out = nn.Linear(hid_dim, vocab_size)
        self.drop = nn.Dropout(dropout)

    def forward(self, prev_token, hidden, enc_out, src_mask):
        x = self.drop(self.emb(prev_token)).unsqueeze(1)
        h, hidden = self.rnn(x, hidden)
        h = h.squeeze(1)
        if self.attention == 'luong':
            scores = torch.bmm(enc_out, self.attn(h).unsqueeze(2)).squeeze(2)
            scores = scores.masked_fill(~src_mask, -1e9)
            ctx = torch.bmm(F.softmax(scores, dim=1).unsqueeze(1), enc_out).squeeze(1)
            h = torch.tanh(self.combine(torch.cat([h, ctx], dim=1)))
        return self.out(self.drop(h)), hidden


class Seq2Seq(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, emb_dim=128, hid_dim=256,
                 n_layers=2, dropout=0.2, rnn='gru', attention='luong', bidirectional=False):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn, bidirectional)
        self.decoder = Decoder(tgt_vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn, attention)

    def encode(self, src, src_len):
        enc_out, hidden = self.encoder(src, src_len)
        return (hidden, enc_out, src != PAD)  # (hidden, enc_out, mask)

    def decode_step(self, prev_token, state):
        hidden, enc_out, mask = state
        logits, hidden = self.decoder(prev_token, hidden, enc_out, mask)
        return logits, (hidden, enc_out, mask)

    def forward(self, src, src_len, tgt, teacher_forcing=0.5):
        # tgt[:, 1:] 예측
        state = self.encode(src, src_len)
        token, outputs = tgt[:, 0], []
        for t in range(1, tgt.size(1)):
            logits, state = self.decode_step(token, state)
            outputs.append(logits)
            use_tf = random.random() < teacher_forcing
            token = tgt[:, t] if use_tf else logits.argmax(1)
        return torch.stack(outputs, dim=1)


def reorder_state(state, idx):
    # beam search용 배치 재정렬
    hidden, enc_out, mask = state
    if isinstance(hidden, tuple):
        hidden = tuple(h.index_select(1, idx) for h in hidden)
    else:
        hidden = hidden.index_select(1, idx)
    return hidden, enc_out.index_select(0, idx), mask.index_select(0, idx)


model = Seq2Seq(len(src_vocab), len(tgt_vocab), **CONFIG).to(DEVICE)
print('params:', sum(p.numel() for p in model.parameters()))

## 학습, greedy / beam 디코딩

In [ ]:
@torch.no_grad()
def greedy_batch(model, src, src_len, max_len=None):
    # batch greedy
    max_len = max_len or getattr(model, 'max_len', MAX_LEN)  # 로드한 모델은 저장된 max_len 사용
    model.eval()
    state = model.encode(src, src_len)
    token = torch.full((src.size(0),), SOS, dtype=torch.long, device=src.device)
    done = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
    out = []
    for _ in range(max_len):
        logits, state = model.decode_step(token, state)
        token = logits.argmax(1)
        out.append(token)
        done |= token == EOS
        if done.all():
            break
    return torch.stack(out, dim=1)


def _to_tensor(text, vocab):
    ids = vocab.encode(text)
    return torch.tensor([ids], device=DEVICE), torch.tensor([len(ids)])


def greedy_decode(model, src_text, src_vocab, tgt_vocab, max_len=None):
    src, src_len = _to_tensor(src_text, src_vocab)
    return tgt_vocab.decode(greedy_batch(model, src, src_len, max_len)[0].tolist())


@torch.no_grad()
def beam_decode(model, src_text, src_vocab, tgt_vocab, beam=BEAM_SIZE, max_len=None):
    max_len = max_len or getattr(model, 'max_len', MAX_LEN)
    model.eval()
    src, src_len = _to_tensor(src_text, src_vocab)
    state = model.encode(src, src_len)
    beams = [([], 0.0)]  # (seq, log prob)
    finished = []
    for _ in range(max_len):
        tokens = torch.tensor([b[0][-1] if b[0] else SOS for b in beams], device=DEVICE)
        if len(beams) != state[1].size(0):
            state = reorder_state(state, torch.zeros(len(beams), dtype=torch.long, device=DEVICE))
        logits, state = model.decode_step(tokens, state)
        logp = F.log_softmax(logits, dim=1)
        cand = []
        for i, (seq, score) in enumerate(beams):
            top = logp[i].topk(beam)
            for lp, tok in zip(top.values.tolist(), top.indices.tolist()):
                cand.append((seq + [tok], score + lp, i))
        cand.sort(key=lambda c: c[1], reverse=True)
        beams, keep = [], []
        for seq, score, i in cand:
            if seq[-1] == EOS:
                finished.append((seq, score / len(seq)))  # 길이 정규화
            else:
                beams.append((seq, score)); keep.append(i)
            if len(beams) == beam:
                break
        if not beams or len(finished) >= beam:
            break
        state = reorder_state(state, torch.tensor(keep, device=DEVICE))
    if not finished:
        finished = [(seq, score / len(seq)) for seq, score in beams]
    best = max(finished, key=lambda f: f[1])[0]
    return tgt_vocab.decode(best)


@torch.no_grad()
def evaluate(model, loader, tgt_vocab):
    # exact match 기준
    model.eval()
    total_loss, n_tok, correct, n = 0.0, 0, 0, 0
    for src, src_len, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        logits = model(src, src_len, tgt, teacher_forcing=1.0)
        gold = tgt[:, 1:]
        total_loss += F.cross_entropy(logits.reshape(-1, logits.size(-1)), gold.reshape(-1),
                                      ignore_index=PAD, reduction='sum').item()
        n_tok += (gold != PAD).sum().item()
        pred = greedy_batch(model, src, src_len)
        for p, g in zip(pred.tolist(), gold.tolist()):
            correct += tgt_vocab.decode(p) == tgt_vocab.decode(g)
            n += 1
    return {'loss': total_loss / n_tok, 'exact_match': correct / n}


def train(model, train_loader, valid_loader, epochs=EPOCHS, lr=LR, save_path=None):
    # best valid acc 모델 복원
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history, best, best_state = [], -1, None
    for ep in range(1, epochs + 1):
        model.train()
        t0, total = time.time(), 0.0
        for src, src_len, tgt in train_loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            logits = model(src, src_len, tgt, TEACHER_FORCING)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1), ignore_index=PAD)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item()
        val = evaluate(model, valid_loader, tgt_vocab)
        history.append({'epoch': ep, 'train_loss': total / len(train_loader), **{f'valid_{k}': v for k, v in val.items()}})
        print(f"epoch {ep:2d} | train loss {history[-1]['train_loss']:.4f} | valid loss {val['loss']:.4f} "
              f"| valid acc {val['exact_match']*100:.2f}% | {time.time()-t0:.0f}s")
        if val['exact_match'] > best:
            best, best_state = val['exact_match'], {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f'best valid acc: {best*100:.2f}%')
    return history

In [ ]:
# 동일 seed로 모델 초기화와 DataLoader 순서를 맞춰 비교합니다.
experiments = {}
for bidirectional in (False, True):
    set_seed(SEED)
    config = dict(CONFIG, bidirectional=bidirectional)
    candidate = Seq2Seq(len(src_vocab), len(tgt_vocab), **config).to(DEVICE)
    label = 'bidirectional' if bidirectional else 'unidirectional'
    print(f'\n=== {label} ===', flush=True)
    started = time.time()
    history = train(candidate, train_loader, valid_loader, EPOCHS, LR)
    experiments[label] = dict(config=config, history=history,
        best_valid=max(h['valid_exact_match'] for h in history),
        seconds=time.time()-started,
        parameters=sum(p.numel() for p in candidate.parameters()),
        state={k: v.detach().cpu().clone() for k,v in candidate.state_dict().items()})
    # 중간 결과를 보존하여 긴 학습 도중 중단돼도 복원할 수 있게 합니다.
    torch.save(experiments, os.path.join(SAVE_DIR, 'en2num_experiments.pt'))
    del candidate
selected = max(experiments, key=lambda name: experiments[name]['best_valid'])
CONFIG = experiments[selected]['config']
history = experiments[selected]['history']
model = Seq2Seq(len(src_vocab), len(tgt_vocab), **CONFIG).to(DEVICE)
model.load_state_dict(experiments[selected]['state'])
print('validation 기준 선택:', selected)


## test 평가

In [ ]:
@torch.no_grad()
def predict_all(model, loader, tgt_vocab):
    rows, i = [], 0
    for src, src_len, _ in loader:
        pred = greedy_batch(model, src.to(DEVICE), src_len)
        for p in pred.tolist():
            s, t, k = loader.dataset.pairs[i]
            rows.append((s, t, tgt_vocab.decode(p), k)); i += 1
    return rows


def exact_match(rows):
    return sum(r[1] == r[2] for r in rows) / len(rows)


def acc_by_length(rows):
    stat = defaultdict(lambda: [0, 0])
    for _, gold, pred, key in rows:
        stat[len(key)][0] += gold == pred
        stat[len(key)][1] += 1
    print('자릿수 | 개수 | 정확도')
    for L in sorted(stat):
        c, n = stat[L]
        print(f'{L:6d} | {n:6d} | {c/n*100:6.2f}%')


def show_examples(rows, n=5):
    right = [r for r in rows if r[1] == r[2]][:n]
    wrong = [r for r in rows if r[1] != r[2]][:n]
    for title, group in (('맞은 예시', right), ('틀린 예시', wrong)):
        print(f'--- {title} ---')
        for s, gold, pred, _ in group:
            print(f'입력: {s}\n정답: {gold}\n예측: {pred}\n')


print('모델 | parameters | best valid | test greedy | seconds')
for label, result in experiments.items():
    candidate = Seq2Seq(len(src_vocab), len(tgt_vocab), **result['config']).to(DEVICE)
    candidate.load_state_dict(result['state'])
    rows = predict_all(candidate, test_loader, tgt_vocab)
    result['test_exact_match'] = exact_match(rows)
    print(f"{label} | {result['parameters']:,} | {result['best_valid']:.4%} | {result['test_exact_match']:.4%} | {result['seconds']:.1f}")
    acc_by_length(rows)
    show_examples(rows)
    del candidate
test_rows = predict_all(model, test_loader, tgt_vocab)
print('최종 선택:', selected, 'test:', exact_match(test_rows))


In [ ]:
# greedy vs beam (test 200개)
sample = test_loader.dataset.pairs[:200]
g = sum(greedy_decode(model, s, src_vocab, tgt_vocab) == t for s, t, _ in sample)
b = sum(beam_decode(model, s, src_vocab, tgt_vocab, BEAM_SIZE) == t for s, t, _ in sample)
print(f'test 200개: greedy {g/len(sample)*100:.1f}% | beam({BEAM_SIZE}) {b/len(sample)*100:.1f}%')

## 저장

In [ ]:
def save_model(path, model, config, src_vocab, tgt_vocab, metrics=None):
    # 저장 형식 고정 (통합 시 로드)
    torch.save({
        'model': model.state_dict(),
        'config': config,
        'src_lang': src_vocab.lang, 'tgt_lang': tgt_vocab.lang,
        'src_vocab': src_vocab.itos, 'tgt_vocab': tgt_vocab.itos,
        'max_len': MAX_LEN,
        'metrics': metrics or {},
    }, path)
    print(f'saved: {path} ({os.path.getsize(path)/1e6:.1f} MB)')


def load_model(path):
    ckpt = torch.load(path, map_location=DEVICE)
    src_vocab = Vocab.from_itos(ckpt['src_lang'], ckpt['src_vocab'])
    tgt_vocab = Vocab.from_itos(ckpt['tgt_lang'], ckpt['tgt_vocab'])
    m = Seq2Seq(len(src_vocab), len(tgt_vocab), **ckpt['config']).to(DEVICE)
    m.load_state_dict(ckpt['model'])
    m.max_len = ckpt['max_len']
    m.eval()
    return m, src_vocab, tgt_vocab, ckpt


SAVE_PATH = os.path.join(SAVE_DIR, f'{SRC}2{TGT}.pt')
save_model(SAVE_PATH, model, CONFIG, src_vocab, tgt_vocab,
           metrics={'test_exact_match': exact_match(test_rows), 'history': history, 'selected': selected,
                    'comparison': {k: {a:b for a,b in v.items() if a != 'state'} for k,v in experiments.items()}})

## 저장된 모델 사용법
위 정의 셀을 실행한 후 마지막 셀에서 저장된 모델을 다시 불러옵니다.
`en2num('four thousand five hundred seven')`처럼 영어 문자열을 입력하면 숫자 문자열을 반환합니다.
통합 노트북에서는 `bidirectional` 옵션과 Encoder projection 코드를 함께 반영해야 합니다.


In [ ]:
m, sv, tv, ckpt = load_model(SAVE_PATH)
print(f"loaded {ckpt['src_lang']} -> {ckpt['tgt_lang']} | test acc {ckpt['metrics']['test_exact_match']*100:.2f}%\n")
for s, t, _ in test_loader.dataset.pairs[:5]:
    print(f'입력 {s!r}\n  greedy: {greedy_decode(m, s, sv, tv)}\n  beam  : {beam_decode(m, s, sv, tv)}\n  정답  : {t}')
def en2num(text):
    return greedy_decode(m, text, sv, tv)

# 저장 전후 예측 일치 확인
for s, _, _ in test_loader.dataset.pairs[:20]:
    assert greedy_decode(model, s, src_vocab, tgt_vocab) == en2num(s)
print('저장/복원 예측 일치: PASS')
print("en2num('four thousand five hundred seven'):", en2num('four thousand five hundred seven'))
